# Comparison between Edgegram and Tokenizer: Efficiency comparison

This notebook compares two tokenization strategies on multilingual text:

- **Word Tokenizer**: Splits on whitespace;each word is one token. Vocabulary is built from the corpus.
- **Edgegram tokenizer**: Splits each word into fixed-length prefix and/or suffix n-grams (e.g. n=3). Useful for subword level features with a compact,reusuable vocabulary.


## Objectives

1. Load the same sample data used in the main tokenization playground.
2. Build both tokenizers on the same corpus and compare.
    - **Vocabulary size**,
    - **Tokens per text** (avg, min, max) by language
    - **Encode speed** (throughput)
3. Show side-by-side tokenization examples.
4. Visualize metrics and document which strategy is more efficient for sequence length and vocab size in this setup.

## Metrics

- **Efficiency** here means: smaller vocabulary and/or shorter sequences (fewer tokens per text) when possible, plus encode speed.

## 1. Imports

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Project paths (notebook lives in notebooks/, project root is parent)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Imports from project
from config import SAMPLE_ES, SAMPLE_EN, SAMPLE_FR
from compare import summarize_tokenizers, collect_examples
from text_tokenizer import EdgegramTokenizer, WordTokenizer

pd.set_option("display.max_colwidth", 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: c:\Users\rafae\Desktop\LLM_practices\tokenization_playgroud
SRC_DIR: c:\Users\rafae\Desktop\LLM_practices\tokenization_playgroud\src


## Load sample data
we load one sentence per line from the sample files as the main playground(`sample_es.txt`, `sample_en.txt`, `sample_fr.txt`)

In [4]:
def load_lines(path):
    if not path.exists():
        raise FileNotFoundError(f"File {path} does not exist")
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

In [5]:
text_es = load_lines(SAMPLE_ES)
text_en = load_lines(SAMPLE_EN)
text_fr = load_lines(SAMPLE_FR)
samples_by_lang = { "es": text_es, "en": text_en, "fr": text_fr }
all_texts = text_es + text_fr + text_en

In [6]:
for lang, texts in samples_by_lang.items():
    print(f"{lang}: {len(texts)} lines")
print(f"Total texts: {len(all_texts)}")

es: 4 lines
en: 4 lines
fr: 4 lines
Total texts: 12


## 2. Initialize tokenizers and build vocabularies.
- **EdgegramTokenizer**: We use `n=3` with both prefix and suffix. Its vocabulary is built dynamically; we run one pass over the full corpus with `encode(..., add_to_vocab=True)` so both tokenizers see the same data.
- **WordTokenizer**: Built from the full corpus with `WordTokenizer.from_corpus(all_texts)` (lowercase by default).



In [7]:
word_tokenizer = WordTokenizer.from_corpus(all_texts)

In [8]:
edge_tokenizer = EdgegramTokenizer(n=3, use_prefix=True, use_suffix=True)

In [10]:
from text_tokenizer import edgegram_tokenizer


for text in all_texts:
    edge_tokenizer.encode(text, add_to_vocab=True)

tokenizers = {
    "word": word_tokenizer,
    "edgegram": edgegram_tokenizer
}


In [11]:
print("Word tokenizer vocab size:", word_tokenizer.get_vocab_size())

Word tokenizer vocab size: 112


In [12]:
print("Edgegram tokenizer vocab size:", edgegram_tokenizer.get_vocab_size())

AttributeError: module 'text_tokenizer.edgegram_tokenizer' has no attribute 'get_vocab_size'